# Pandas vs PySpark — Side-by-Side Demo

This notebook provides a learner-friendly, side-by-side comparison of the same operations in both frameworks, plus performance benchmark visualisations.

**Sections:**
1. Environment setup & synthetic data
2. Reading & inspecting data
3. Filtering
4. GroupBy aggregation
5. Window functions
6. Joining DataFrames
7. Performance benchmarks (vectorised vs iterrows, Pandas UDF vs Python UDF)
8. Memory optimisation visualisation

## 1. Setup

In [ ]:
import sys, os, time, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yaml

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Locate project root
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
CONFIG_PATH  = PROJECT_ROOT / 'config.yaml'
SRC_PATH     = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Logging
from config_loader import setup_logging
setup_logging(cfg)
logger = logging.getLogger('notebook')

SEED  = cfg['data']['random_seed']
N     = cfg['data']['num_rows']
rng   = np.random.default_rng(SEED)

print(f'Project root : {PROJECT_ROOT}')
print(f'Config loaded: seed={SEED}, N={N}')

## 2. Synthetic Data Generation

In [ ]:
# ─── Pandas DataFrame ───────────────────────────────────────────────────────
dates    = pd.date_range('2022-01-01', periods=N, freq='h')
regions  = rng.choice(['North','South','East','West'], N)
products = rng.choice(['Widget','Gadget','Doohickey'], N)
revenues = rng.exponential(500, N).round(2)
units    = rng.integers(1, 100, N)

pdf = pd.DataFrame({
    'timestamp'     : dates,
    'region'        : regions,
    'product'       : products,
    'salesperson_id': rng.integers(1, 21, N).astype(int),
    'revenue'       : revenues,
    'units'         : units,
})

print('Pandas DataFrame:')
display(pdf.head(5))
print(f'Shape: {pdf.shape}  |  Memory: {pdf.memory_usage(deep=True).sum()/1e6:.1f} MB')

In [ ]:
# ─── Spark Session ──────────────────────────────────────────────────────────
from pyspark_core.spark_session import build_spark_session
spark = build_spark_session(cfg)
print(f'Spark version: {spark.version}')

# Create Spark DataFrame from pandas
sdf = spark.createDataFrame(pdf)
print(f'Spark DataFrame: {sdf.count()} rows, {len(sdf.columns)} columns')
sdf.printSchema()

## 3. Filtering

In [ ]:
# ─── Pandas ─────────────────────────────────────────────────────────────────
pd_filtered = pdf[(pdf['revenue'] > 1000) & (pdf['region'] == 'North')]
print(f'[Pandas]  Filter revenue>1000 AND region=North: {len(pd_filtered):,} rows')

# ─── PySpark ────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
sp_filtered = sdf.filter((F.col('revenue') > 1000) & (F.col('region') == 'North'))
print(f'[PySpark] Filter revenue>1000 AND region=North: {sp_filtered.count():,} rows')

## 4. GroupBy Aggregation

In [ ]:
# ─── Pandas ─────────────────────────────────────────────────────────────────
pd_agg = (
    pdf.groupby(['region', 'product'])
    .agg(
        total_revenue=('revenue', 'sum'),
        mean_revenue =('revenue', 'mean'),
        txn_count    =('revenue', 'count'),
    )
    .round(2)
    .reset_index()
)
print('[Pandas] GroupBy result:')
display(pd_agg.sort_values('total_revenue', ascending=False).head(6))

# ─── PySpark ────────────────────────────────────────────────────────────────
sp_agg = (
    sdf.groupBy('region', 'product')
    .agg(
        F.round(F.sum('revenue'),   2).alias('total_revenue'),
        F.round(F.avg('revenue'),   2).alias('mean_revenue'),
        F.count('*')                  .alias('txn_count'),
    )
    .orderBy(F.desc('total_revenue'))
)
print('[PySpark] GroupBy result:')
sp_agg.show(6, truncate=False)

In [ ]:
# Visualise GroupBy result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot = pd_agg.pivot(index='region', columns='product', values='total_revenue')
pivot.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='white', linewidth=0.5)
axes[0].set_title('Total Revenue by Region & Product (Pandas GroupBy)', fontsize=12)
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Product')

pd_agg.groupby('region')['txn_count'].sum().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
    colors=['#4C72B0','#DD8452','#55A868','#C44E52']
)
axes[1].set_title('Transaction Share by Region', fontsize=12)
axes[1].set_ylabel('')

plt.tight_layout()
out_dir = PROJECT_ROOT / cfg['data']['output_dir']
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'groupby_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved groupby_chart.png')

## 5. Window Functions

In [ ]:
window_size = cfg['window']['rolling_window']

# ─── Pandas rolling ─────────────────────────────────────────────────────────
ts = pdf.set_index('timestamp').sort_index()
ts[f'sma_{window_size}h'] = ts['revenue'].rolling(window_size).mean()
ts['cumrev'] = ts['revenue'].cumsum()
print(f'[Pandas] Rolling SMA({window_size}) and cumulative revenue added')

# ─── PySpark window ─────────────────────────────────────────────────────────
from pyspark.sql import Window
win = Window.orderBy('timestamp').rowsBetween(-(window_size - 1), 0)
win_cum = Window.orderBy('timestamp').rowsBetween(Window.unboundedPreceding, 0)
sp_win = (
    sdf
    .withColumn(f'sma_{window_size}h', F.round(F.avg('revenue').over(win), 2))
    .withColumn('cumrev', F.round(F.sum('revenue').over(win_cum), 2))
)
print('[PySpark] Window function columns added:')
sp_win.select('timestamp', 'revenue', f'sma_{window_size}h', 'cumrev').show(5, truncate=False)

In [ ]:
# Visualise rolling vs actual
sample = ts['2022-01-01':'2022-01-15'][['revenue', f'sma_{window_size}h']]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample.index, sample['revenue'],          alpha=0.4, label='Hourly Revenue', color='steelblue', linewidth=1)
ax.plot(sample.index, sample[f'sma_{window_size}h'], color='crimson',  label=f'SMA({window_size}h)',    linewidth=2)
ax.set_title(f'Revenue with SMA({window_size}) — 2 Week Window (Pandas rolling)', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
ax.legend()
plt.tight_layout()
plt.savefig(out_dir / 'rolling_window_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Joining DataFrames

In [ ]:
# Employee lookup
emp_pdf = pd.DataFrame({
    'salesperson_id': range(1, 21),
    'name'          : [f'Employee_{i}' for i in range(1, 21)],
    'department'    : rng.choice(['Sales','Marketing','Operations'], 20).tolist(),
})

# ─── Pandas join ────────────────────────────────────────────────────────────
pd_joined = pdf.merge(emp_pdf, on='salesperson_id', how='left')
print(f'[Pandas] Left join: {len(pd_joined):,} rows | dept sample:')
display(pd_joined[['region','product','revenue','name','department']].head(4))

# ─── PySpark broadcast join ─────────────────────────────────────────────────
emp_sdf = spark.createDataFrame(emp_pdf)
sp_joined = sdf.join(F.broadcast(emp_sdf), on='salesperson_id', how='left')
print(f'[PySpark] Broadcast left join: {sp_joined.count():,} rows')
sp_joined.select('region','product','revenue','name','department').show(4, truncate=False)

## 7. Performance Benchmarks

In [ ]:
# Vectorised vs iterrows benchmark
bench_n   = 10_000
bench_pdf = pdf.head(bench_n).reset_index(drop=True)

# iterrows
t0 = time.perf_counter()
results_iter = []
for _, row in bench_pdf.iterrows():
    results_iter.append(row['revenue'] * 1.15 - row['units'] * 2)
t_iter = time.perf_counter() - t0

# itertuples
t0 = time.perf_counter()
results_ituples = [r.revenue * 1.15 - r.units * 2 for r in bench_pdf.itertuples(index=False)]
t_ituples = time.perf_counter() - t0

# vectorised
t0 = time.perf_counter()
results_vec = bench_pdf['revenue'] * 1.15 - bench_pdf['units'] * 2
t_vec = time.perf_counter() - t0

print(f'N={bench_n:,} rows')
print(f'  iterrows  : {t_iter:.4f}s')
print(f'  itertuples: {t_ituples:.4f}s')
print(f'  vectorised: {t_vec:.6f}s  ({t_iter/t_vec:.0f}x faster than iterrows)')

In [ ]:
# Benchmark bar chart
methods = ['iterrows', 'itertuples', 'vectorised']
times   = [t_iter, t_ituples, t_vec]
colors  = ['#e06c75', '#e5c07b', '#98c379']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, times, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, labels=[f'{t:.4f}s' for t in times], padding=3, fontsize=10)
ax.set_yscale('log')
ax.set_title(f'Pandas Row Iteration vs Vectorised (N={bench_n:,})', fontsize=13)
ax.set_ylabel('Time (seconds, log scale)')
ax.set_xlabel('Method')
plt.tight_layout()
plt.savefig(out_dir / 'benchmark_iteration.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Memory Optimisation

In [ ]:
from pandas_optimization.dtype_optimizer import CategoryOptimizer, NumericDowncaster, MemoryReporter

cat_threshold = cfg['pandas']['category_threshold']
mem_reporter  = MemoryReporter()

mb_orig     = mem_reporter.report(pdf, 'original')
pdf_cat     = CategoryOptimizer(threshold=cat_threshold).optimize(pdf)
mb_cat      = mem_reporter.report(pdf_cat, 'after category')
pdf_final   = NumericDowncaster().optimize(pdf_cat)
mb_final    = mem_reporter.report(pdf_final, 'after downcast')

print(f'\nMemory: {mb_orig:.2f} MB -> {mb_cat:.2f} MB -> {mb_final:.2f} MB')
print(f'Total saving: {(1 - mb_final/mb_orig)*100:.1f}%')

In [ ]:
# Memory optimisation waterfall chart
labels  = ['Original', 'After Category\nDtype', 'After Numeric\nDowncast']
mem_mbs = [mb_orig, mb_cat, mb_final]
colors  = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, mem_mbs, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
ax.bar_label(bars, labels=[f'{m:.2f} MB' for m in mem_mbs], padding=4, fontsize=10)
ax.set_title(f'Memory Usage After Dtype Optimisation (N={N:,})', fontsize=13)
ax.set_ylabel('Memory (MB)')
ax.set_ylim(0, max(mem_mbs) * 1.2)

for i in range(1, len(mem_mbs)):
    saving = (mem_mbs[i-1] - mem_mbs[i]) / mem_mbs[i-1] * 100
    ax.annotate(
        f'-{saving:.1f}%',
        xy=(i, mem_mbs[i] + max(mem_mbs)*0.02),
        ha='center', color='crimson', fontsize=11, fontweight='bold'
    )

plt.tight_layout()
plt.savefig(out_dir / 'memory_optimisation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# dtype breakdown per column
def get_col_memory(df, label):
    return (
        df.memory_usage(deep=True)
          .drop('Index')
          .rename(label)
          / 1024**2
    ).round(3)

mem_df = pd.concat([
    get_col_memory(pdf,       'Original'),
    get_col_memory(pdf_cat,   'Category'),
    get_col_memory(pdf_final, 'Downcast'),
], axis=1)

mem_df.plot(kind='barh', figsize=(10, 6), colormap='Set2', edgecolor='white')
plt.title('Per-Column Memory Usage (MB) Before and After Optimisation', fontsize=12)
plt.xlabel('Memory (MB)')
plt.tight_layout()
plt.savefig(out_dir / 'per_column_memory.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Pandas vs PySpark Comparison Table

In [ ]:
comparison = pd.DataFrame([
    ('Execution',        'Eager (immediate)',              'Lazy (DAG, triggered by action)',     'Eager (default) / Lazy (.lazy())'),
    ('Scale',           'Single machine',                 'Distributed cluster',                  'Single machine (Rust, multi-threaded)'),
    ('Language',        'Python',                         'Python (JVM backend)',                  'Python / Rust'),
    ('Memory model',    'In-memory (RAM)',                 'In-memory + disk spill',               'In-memory (columnar, cache-efficient)'),
    ('Mutability',      'Mutable (copy-on-write in pd2)', 'Immutable (functional transforms)',    'Immutable'),
    ('Join syntax',     'df.merge(other, on=..., how=...)', 'df.join(other, on=..., how=...)',    'df.join(other, on=..., how=...)'),
    ('GroupBy',         'df.groupby().agg()',              'df.groupBy().agg()',                   'df.group_by().agg()'),
    ('Window fn',       'df.rolling() / .expanding()',    'Window.partitionBy().orderBy()',        'pl.col(...).rolling_mean()'),
    ('UDFs',            'Native Python functions',        'Python UDF or Pandas UDF (Arrow)',     'Native Rust expressions (fast)'),
    ('SQL support',     'pandasql / DuckDB',              'spark.sql("SELECT ...")',               'pl.SQLContext'),
    ('Schema enforce',  'Loose (inferred)',               'Strict (StructType)',                   'Strict (inferred / declared)'),
    ('Streaming',       'No (batch only)',                'Structured Streaming',                 'No (batch only)'),
    ('Best for',        '< 1 GB, exploratory analysis',  '> 1 GB, ETL, distributed ML',          '< 50 GB, fast local transforms'),
], columns=['Concept', 'Pandas', 'PySpark', 'Polars'])

display(comparison.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'font-size': '11px'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2d3748'), ('color', 'white'), ('font-size', '12px')]},
    {'selector': 'tr:nth-of-type(even)', 'props': [('background-color', '#f7f7f7')]},
]))

In [ ]:
# Cleanup Spark session
spark.stop()
print('SparkSession stopped. Notebook complete.')